# 04 — Inference & Grad-CAM

**But** : charger le meilleur checkpoint et effectuer des prédictions individuelles, avec interprétabilité Grad-CAM.

> Ce notebook est **totalement indépendant** — il recharge le modèle depuis le disque.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/LAAFI_AI
except ImportError:
    pass

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import datasets
datasets.logging.set_verbosity_error()

import numpy as np
import torch
import matplotlib.pyplot as plt

from laafi_ai.cli_train import set_seed
from laafi_ai.config import ExperimentConfig
from laafi_ai.data import PCamDataModule
from laafi_ai.gradcam import make_gradcam_overlay
from laafi_ai.inference import load_model_from_checkpoint
from laafi_ai.logging_utils import setup_logging
from laafi_ai.model import get_device
from laafi_ai.paths import resolve_project_paths

setup_logging()

## 2. Configuration et chargement du modèle

In [ ]:
config = ExperimentConfig.from_yaml('configs/default.yaml')
config.output_dir = 'outputs_finetune_layer4'

set_seed(config.seed)
device = get_device(config.device)
paths = resolve_project_paths(config, project_root=PROJECT_ROOT)

BEST_CHECKPOINT = paths.checkpoint_dir / 'best_resnet50_pcam.pt'
model, loaded_config = load_model_from_checkpoint(BEST_CHECKPOINT, device)
print('Modèle chargé ✓')

## 3. Charger quelques images du test set

In [ ]:
data_module = PCamDataModule(config.data)
_, _, test_loader = data_module.dataloaders()

# Prendre un batch pour la démo
images, labels = next(iter(test_loader))
print(f'Batch de {images.shape[0]} images chargé.')

## 4. Prédictions sur le batch

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(images.to(device)).squeeze(1)
    probs = torch.sigmoid(logits).cpu().numpy()

threshold = config.training.decision_threshold
preds = (probs >= threshold).astype(int)

for i in range(min(8, len(probs))):
    label_str = 'Métastase' if labels[i] == 1 else 'Normal'
    pred_str = 'Métastase' if preds[i] == 1 else 'Normal'
    marker = '✓' if preds[i] == int(labels[i]) else '✗'
    print(f'  [{marker}] Image {i}: vrai={label_str}, prédit={pred_str} (p={probs[i]:.4f})')

## 5. Grad-CAM — Interprétabilité

Grad-CAM permet de voir *quelles régions de l'image* ont le plus influencé la prédiction du modèle.

In [ ]:
# Couche cible pour Grad-CAM (dernier bloc convolutif de ResNet50)
target_layer = model.layer4[-1]

N_SAMPLES = min(4, len(images))
fig, axes = plt.subplots(N_SAMPLES, 2, figsize=(8, 4 * N_SAMPLES))

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(N_SAMPLES):
    # Image originale dé-normalisée
    img_np = images[i].permute(1, 2, 0).numpy()
    img_rgb = np.clip(img_np * std + mean, 0, 1)

    # Grad-CAM overlay
    overlay = make_gradcam_overlay(model, images[i], target_layer)

    label_str = 'Métastase' if labels[i] == 1 else 'Normal'
    pred_str = f'p={probs[i]:.3f}'

    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f'Vrai: {label_str}')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title(f'Grad-CAM ({pred_str})')
    axes[i, 1].axis('off')

plt.suptitle('Grad-CAM — Zones d\'attention du modèle', fontsize=13)
plt.tight_layout()
plt.savefig(paths.figures_dir / 'gradcam_samples.png', dpi=150)
plt.show()
print('Sauvegardé dans', paths.figures_dir / 'gradcam_samples.png')

---
**Fin du pipeline.** Pour un entraînement complet, modifiez les paramètres dans `02_train.ipynb` et relancez.